# 二维微生物资源世界：神经回路与两个鞭毛
目标：在平面中寻找并吞食随机资源点。每个资源点的浓度按距离指数衰减，低于阈值截断。个体头部左、右各有一个浓度感受器，尾部左、右各有一个鞭毛；每个鞭毛由一对运动神经元控制。控制网络有 32 个神经元与稀疏局部连接，无可寻址记忆或矩阵查询。

本笔记本先训练，再对未见过的世界种子评估，最后播放轨迹。训练只改长期连接权重、偏置和神经元更新速度；运行时每步只看到当前两侧感受器的浓度。


In [ ]:
from pathlib import Path
import subprocess
repo = Path('/content/Parrot')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/DoctorAwe/Parrot.git', str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
commit = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('代码版本:', commit)
%cd /content/Parrot


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
result_dir = Path('/content/drive/MyDrive/Parrot/microbe_results')
result_dir.mkdir(parents=True, exist_ok=True)


## 快速检查
先运行一个很短的场景，检查世界、神经回路和轨迹记录。


In [ ]:
from microbe_world import Config, SparseCircuit, rollout
small = Config(steps=30)
circuit_check = SparseCircuit(small)
reward_check, replay_check = rollout(circuit_check, circuit_check.reflex_genome(), 123, small, record=True)
print('神经元:', circuit_check.n, '突触:', circuit_check.edge_count, '帧数:', len(replay_check['trajectory']))
assert circuit_check.n == 32 and len(replay_check['trajectory']) == 30


## 训练
从一个简单的左右趋化反射连接出发，进化搜索稀疏回路的连接权重和神经元时间常数。每代使用相同的 12 个训练世界；每代结束都会把当前最佳回路保存到 Drive。默认约 35 代，CPU 也可运行，耗时依 Colab 机器而异。可以先把 `generations` 改成 5 检查流程。


In [ ]:
from microbe_world import train
cfg = Config(steps=320, resource_count=18, neuron_count=32)
circuit, genome, history = train(generations=35, population=32,
                                 seeds=range(1000, 1012), config=cfg, out=result_dir)
print('最后一代:', history[-1])


## 未见世界评估
先在单独的选择世界中比较进化回路与起始反射，选出最终展示回路；再在另一批从未参与训练或选择的 40 个世界中报告表现。主指标是平均吃到的资源点数；训练奖励还包含很小的浓度变化引导项。


In [ ]:
import json
import numpy as np
from microbe_world import evaluate
selection_seeds = range(2000, 2020)
reflex_genome = circuit.reflex_genome()
selection = {
    'evolved': evaluate(circuit, genome, selection_seeds),
    'starting_reflex': evaluate(circuit, reflex_genome, selection_seeds),
}
selected_name = max(selection, key=lambda name: selection[name]['mean_eaten'])
selected_genome = genome if selected_name == 'evolved' else reflex_genome
np.save(result_dir / 'selected_genome.npy', selected_genome)
test_seeds = range(3000, 3040)
rng = np.random.default_rng(678)
random_scores = [evaluate(circuit, circuit.random_genome(rng), test_seeds)['mean_eaten']
                 for _ in range(10)]
comparison = {
    'selection': selection, 'selected_name': selected_name,
    'evolved_test': evaluate(circuit, genome, test_seeds),
    'starting_reflex_test': evaluate(circuit, reflex_genome, test_seeds),
    'selected_test': evaluate(circuit, selected_genome, test_seeds),
    'random_circuits_mean_eaten': float(np.mean(random_scores)),
    'random_circuits_each_mean_eaten': random_scores,
    'git_commit': commit,
}
(result_dir / 'evaluation.json').write_text(json.dumps(comparison, indent=2), encoding='utf-8')
print(json.dumps(comparison, indent=2))


## 可播放的世界
绿色表示尚未吃掉的资源点，灰色表示已吃掉。背景是初始资源浓度；黑色圆点为个体，青色点为左右感受器，蓝/红线为左右两个鞭毛。使用播放键、帧滑块和速度菜单观察运动。下方同时显示感受器读数及已吃掉的资源点数。


In [ ]:
from microbe_world import show_replay
reward, replay = rollout(circuit, selected_genome, seed=3001, config=cfg, record=True)
(result_dir / 'replay.json').write_text(json.dumps(replay), encoding='utf-8')
print('回路:', selected_name, '资源点:', replay['eaten'], '/', cfg.resource_count, '奖励:', round(reward, 2))
show_replay(replay)


## 训练曲线与回传
将结果、最佳回路和轨迹打包。把 ZIP 放回项目 `result/` 目录供后续分析。


In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 3))
plt.plot([h['generation'] for h in history], [h['best_reward'] for h in history])
plt.xlabel('Generation'); plt.ylabel('Best training reward'); plt.grid(True); plt.show()
import shutil
archive = shutil.make_archive(str(result_dir), 'zip', root_dir=str(result_dir.parent), base_dir=result_dir.name)
print('回传文件:', archive)
